# Laya V4.2 — **Replace ALL Full Attention First, THEN All Sliding Attention** with Progressive CeNN Distillation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V42_Progressive_AllFull_Then_AllSliding_CeNN_Colab.ipynb)

### Exact replacement order
**Phase A — Full attention, one-by-one:** **0 → 3 → 6 → 9 → 12 → 15 → 18 → 21 → 24 → 27**. Train a long-range CeNN replacement for each *attention module only*, validate it, save a checkpoint, then train the next layer on activations produced by the current student prefix.

**Phase B — Sliding attention, one-by-one:** **1 → 2 → 4 → 5 → 7 → 8 → 10 → 11 → 13 → 14 → 16 → 17 → 19 → 20 → 22 → 23 → 25 → 26**. Train local CeNN to replace only the attention module using the progressively changed full-attention and sliding-attention prefix.

### Preserve original residual topology, no FFN compression
For EVERY encoder layer, including full and sliding:
\`attn_input = LN_att(hidden)\` → \`attn_out = CeNN(attn_input)\` → \`attn_residual = hidden + attn_out\` → \`FFN(LN_mlp(attn_residual))\` → \`output = attn_residual + FFN_output\`. **ModernBERT's original block performs both residual additions. This notebook replaces \`layer.attn\`, NOT \`layer\`, so both pretrained residuals, normalization modules, pretrained MLP and classifier remain exactly in place.** Verify by forward hooks before training.

The CeNN approximates the original **value mixing and output projection**. It reuses the frozen pretrained attention's V projection and Wo projection, trains lightweight depthwise local and long-range convolutions and an optional gated global summary. Full layers use RF1025 long-range CeNN; sliding layers use a local radius-64 and radius-128 two-scale CeNN. No explicit per-query key routing or duplicate pretrained model.

### Source-derived motivation
V4.1, group 18–20: **95.5%** held-out teacher agreement, **83.5%** gold vs **86.5%** teacher, about **2.12×** speedup for the compressed sliding pair but only **1.02–1.04×** whole-model speed. Training the FFN separately helped substantially; therefore V4.2 **does not compress FFNs at all** during attention research.

### Safe experiment design
- **TRAIN/PROBE/AUDIT are disjoint by case** from source TRAIN. Official TEST untouched.
- At each layer, capture target outputs of the **native attention on inputs from the current compressed prefix**. The frozen native QKV/rotary/SDPA teacher supplies the local target. Train only that layer's small CeNN.
- For the explicit goal of **replacing all 28 attention modules**, default \`SELECTION_MODE="research_all"\` keeps every trained replacement, even when the intermediate quality gate fails. **This is experimental, not deployment-safe**. Switch to \`"quality_gated"\` to reject a trained layer if it lowers whole-model decision quality or does not improve latency.
- After each layer: record local cosine/NMSE, full-model PROBE teacher agreement/gold accuracy/JS, and matched GPU timing; checkpoint the adapter and progress. Resume already-completed layers after a cell rerun or disconnect. Optional Drive-backed storage survives VM deletion.
- Test all full layers as one collective model **before** replacing any sliding layer. Then continue sliding sequentially. At the very end, evaluate the final architecture on the **untouched AUDIT once** with paired CUDA timing, comparison to the original native model, final quality gates and adapter-only checkpoint.
- **No guarantee** of accuracy recovery or speedup from CeNN. A radius-512 receptive field does not reproduce arbitrary full attention. Audit is never used to select configurations.

This notebook runs 28 training stages, so a full run on T4 may take hours. First run with \`STEPS_FULL=80\` and \`STEPS_SLIDING=60\` for architecture screening, then increase steps on an unchanged configuration after inspecting the result.


In [ ]:
#@title 1. Install pinned V4.2 dependencies and GPU check
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4 or better)."
print("CUDA",torch.version.cuda,"GPU",torch.cuda.get_device_name(0),"Torch",torch.__version__)


In [ ]:
#@title 2. Frozen Laya checkpoint + progressive full-then-sliding settings
from pathlib import Path
from contextlib import contextmanager
import math,copy,json,random,hashlib,time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from tinycenn_lm.standalone_decision import (
    QTYPES,build_checkpoint_model,build_sequence,collate_items,
    normalize_question,full_attention_indices,
)
from tinycenn_lm.decision_training import require_finite

SOURCE_MODEL="convaiinnovations/laya-typed-decisions"
SEED=2042
MAX_LEN=512 #@param {type:"integer"}
TRAIN_CASES=500 #@param {type:"integer"}
DEV_CASES=240 #@param {type:"integer"}
BATCH=2 #@param {type:"integer"}
RANK=96 #@param {type:"integer"}
STEPS_FULL=100 #@param {type:"integer"}
STEPS_SLIDING=80 #@param {type:"integer"}
LOCAL_TRAIN_BATCHES=16 #@param {type:"integer"}
LOCAL_PROBE_BATCHES=4 #@param {type:"integer"}
PROBE_DECISIONS=120 #@param {type:"integer"}
AUDIT_DECISIONS=200 #@param {type:"integer"}
LEARNING_RATE=0.0003 #@param {type:"number"}
LOG_EVERY=25 #@param {type:"integer"}
BENCH_REPEATS=3 #@param {type:"integer"}
SELECTION_MODE="research_all" #@param ["research_all", "quality_gated"]
MIN_AGREEMENT=.95 #@param {type:"number"}
MAX_GOLD_LOSS_PP=1.0 #@param {type:"number"}
REQUIRE_SPEEDUP_IN_GATED_MODE=False #@param {type:"boolean"}
RESUME=True #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE=False #@param {type:"boolean"}

OUT=Path("/content/Laya_V42_AllFull_Then_Sliding")
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/V42")
OUT.mkdir(parents=True,exist_ok=True)
assert 1<=BATCH<=4 and 16<=RANK<=256
assert 10<=STEPS_FULL and 10<=STEPS_SLIDING and LOCAL_TRAIN_BATCHES>=2
assert LOG_EVERY>=1 and BENCH_REPEATS>=2 and TRAIN_CASES+DEV_CASES<=1200
assert SELECTION_MODE in ("research_all","quality_gated")
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda")
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
source_dir=Path(snapshot_download(
    SOURCE_MODEL,allow_patterns=[
        "rl_agent_config.json","model.safetensors","encoder/*","tokenizer/*"
    ]
))
teacher,source_cfg=build_checkpoint_model(source_dir)
teacher.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
tokenizer=AutoTokenizer.from_pretrained(str(source_dir/"tokenizer"))
ORIGINAL_LAYERS=tuple(teacher.encoder.layers)
ORIGINAL_ATTN=tuple(layer.attn for layer in ORIGINAL_LAYERS)
FULL=tuple(full_attention_indices(teacher))
SLIDING=tuple(i for i in range(28) if i not in FULL)
assert FULL==(0,3,6,9,12,15,18,21,24,27)
assert len(SLIDING)==18
assert int(teacher.encoder.config.local_attention)//2==64
assert MAX_LEN<=int(source_cfg.get("max_len",MAX_LEN))
ORDER=FULL+SLIDING
print("PHASE A full:",FULL)
print("PHASE B sliding:",SLIDING)
print("One original Laya model frozen. T4 GPU",torch.cuda.get_device_name(0))
print("Selection mode:",SELECTION_MODE,"experimental forced-all" if SELECTION_MODE=="research_all" else "gated")


In [ ]:
#@title 3. Case-disjoint TRAIN / PROBE / AUDIT data + batch utilities
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V4.2 teacher helper")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items)>=max(PROBE_DECISIONS,LOCAL_PROBE_BATCHES*BATCH),"Insufficient probe decisions"
assert len(audit_items)>=AUDIT_DECISIONS,"Insufficient audit decisions"
print("official test remains untouched")


In [ ]:
#@title 4. Attention-only CeNN: pretrained value + output matrices, intact original layer residuals and FFNs
class ValueCeNNAttention(nn.Module):
    """Only replace ModernBERT layer.attn, keeping its two residuals and MLP."""
    def __init__(self,original,full,rank=96):
        super().__init__()
        object.__setattr__(self,"source_attention",original)
        self.full=bool(full)
        self.width=int(original.all_head_size)
        self.down=nn.Linear(self.width,rank,bias=False)
        self.pointwise=nn.Linear(rank,rank,bias=False)
        self.local64=nn.Conv1d(rank,rank,9,dilation=16,padding=64,groups=rank,bias=False)
        self.local128=nn.Conv1d(rank,rank,9,dilation=32,padding=128,groups=rank,bias=False)
        if self.full:
            self.long1025=nn.Conv1d(rank,rank,9,dilation=128,padding=512,groups=rank,bias=False)
            self.global_mix=nn.Linear(rank,rank,bias=False)
        self.up=nn.Linear(rank,self.width,bias=False)
        self.gain=nn.Parameter(torch.tensor(0.))
        self.blend=nn.Parameter(torch.tensor(0.))
        nn.init.normal_(self.up.weight,std=.003)

    def core(self,x,valid):
        mask=valid.unsqueeze(-1)
        D=self.width
        native=self.source_attention
        vb=native.Wqkv.bias[2*D:] if native.Wqkv.bias is not None else None
        values=F.linear(x,native.Wqkv.weight[2*D:3*D],vb)
        values=values*mask.to(values.dtype)
        smooth=F.avg_pool1d(values.transpose(1,2),9,1,4).transpose(1,2)
        z=F.silu(self.down(values))
        conv=z.transpose(1,2).contiguous()
        spatial=(self.local64(conv)+self.local128(conv)).transpose(1,2)
        if self.full:
            spatial=spatial+self.long1025(conv).transpose(1,2)
            denom=mask.float().sum(1,keepdim=True).clamp_min(1)
            pooled=(z*mask.to(z.dtype)).sum(1,keepdim=True)/denom
            spatial=spatial+self.global_mix(pooled)
        delta=self.up(F.silu(z+self.pointwise(z)+spatial))
        alpha=torch.sigmoid(self.blend)
        combined=alpha*values+(1-alpha)*smooth+torch.sigmoid(self.gain)*delta
        combined=combined*mask.to(combined.dtype)
        output=F.linear(combined,native.Wo.weight,native.Wo.bias)
        return output*mask.to(output.dtype)

    def forward(self,hidden_states,attention_mask=None,valid_mask=None,**kwargs):
        if valid_mask is None:
            if attention_mask is None:
                valid=torch.ones(hidden_states.shape[:2],device=hidden_states.device,dtype=torch.bool)
            elif attention_mask.ndim==2:
                valid=attention_mask.bool()
            else:
                assert attention_mask.ndim==4
                edge=attention_mask[:,0,0,:]
                valid=edge if edge.dtype==torch.bool else edge.float()>-1e4
        else:
            valid=valid_mask.bool()
        if self.training:
            with torch.autocast("cuda",enabled=False):
                y=self.core(hidden_states.float(),valid)
        else:
            with torch.autocast("cuda",dtype=amp_dtype):
                y=self.core(hidden_states,valid)
        # This is the exact original attention output contract.
        return (y,)

@contextmanager
def install_plan(plan):
    assert all(teacher.encoder.layers[i].attn is ORIGINAL_ATTN[i] for i in range(28))
    try:
        for i,adapter in plan.items():
            teacher.encoder.layers[i].attn=adapter
        yield teacher
    finally:
        for i in range(28):
            teacher.encoder.layers[i].attn=ORIGINAL_ATTN[i]
        assert all(teacher.encoder.layers[i].attn is ORIGINAL_ATTN[i] for i in range(28))

def new_adapter(i):
    return ValueCeNNAttention(ORIGINAL_ATTN[i],i in FULL,RANK).to(device)
def adapter_state(adapter):
    return {k:v.detach().cpu().clone() for k,v in adapter.state_dict().items()}

# Identity of every original norm and FFN is unchanged by attention swap.
assert all(teacher.encoder.layers[i] is ORIGINAL_LAYERS[i] for i in range(28))
test=new_adapter(18).eval()
with install_plan({18:test}),torch.inference_mode(),torch.autocast("cuda",dtype=amp_dtype):
    logits,_=teacher(**batch(probe_items,0,BATCH))
assert torch.isfinite(logits).all()
assert ORIGINAL_LAYERS[18].attn is ORIGINAL_ATTN[18]
print("V4.2 attention-only forward/rollback/residual-layer identity PASS")
print("Trainable CeNN parameters:",sum(p.numel() for p in test.parameters()))
del test


In [ ]:
#@title 5. Native teacher targets from the CURRENT progressively compressed prefix
def cpu_float(t):
    return t.detach().float().cpu().contiguous()
def to_cuda(record):
    # clone cached tensors outside inference_mode before autograd
    return {k:v.to(device,non_blocking=True).clone() for k,v in record.items()}

@torch.inference_mode()
def capture_layer(i,current,items,n_batches,label):
    assert i not in current
    caches=[]
    with install_plan(current):
        for j in range(n_batches):
            selected=items[j*BATCH:(j+1)*BATCH]
            if len(selected)!=BATCH:break
            b=direct_batch(selected)
            got={}
            def pre(mod,args,kwargs):
                got["x"]=args[0].detach()
            def post(mod,args,kwargs,out):
                got["y"]=out[0].detach()
            h1=ORIGINAL_ATTN[i].register_forward_pre_hook(pre,with_kwargs=True)
            h2=ORIGINAL_ATTN[i].register_forward_hook(post,with_kwargs=True)
            try:
                with torch.autocast("cuda",dtype=amp_dtype):
                    teacher(**b)
            finally:
                h1.remove();h2.remove()
            assert set(got)=={"x","y"},(i,got.keys())
            caches.append(dict(x=cpu_float(got["x"]),y=cpu_float(got["y"]),
                               valid=b["attention_mask"].detach().cpu().bool()))
    assert caches,(label,i,"empty cache")
    return caches

def mse_and_cosine(adapter,row):
    with torch.autocast("cuda",enabled=False):
        out=adapter(row["x"].float(),valid_mask=row["valid"])[0].float()
        goal=row["y"].float()
        mask=row["valid"]
        m=mask[:,:,None].float()
        nmse=((out-goal).square()*m).sum()/(goal.square()*m).sum().clamp_min(1e-8)
        cosine=1-F.cosine_similarity(out[mask],goal[mask],dim=-1).mean()
        return nmse+.15*cosine,nmse,cosine

@torch.inference_mode()
def local_score(adapter,cache):
    adapter.eval()
    sq=power=csum=0.
    total=0
    for raw in cache:
        row=to_cuda(raw)
        with torch.autocast("cuda",dtype=amp_dtype):
            out=adapter(row["x"],valid_mask=row["valid"])[0]
        valid=row["valid"]
        m=valid[:,:,None].float()
        sq+=float(((out.float()-row["y"].float()).square()*m).sum())
        power+=float((row["y"].float().square()*m).sum())
        csum+=float(F.cosine_similarity(
            out.float()[valid],row["y"].float()[valid],dim=-1).sum())
        total+=int(valid.sum())
    return dict(cosine=csum/max(total,1),nmse=sq/max(power,1e-8),tokens=total)

assert len(train_items)>=LOCAL_TRAIN_BATCHES*BATCH
assert len(probe_items)>=max(PROBE_DECISIONS,LOCAL_PROBE_BATCHES*BATCH)
r=capture_layer(FULL[0],{},probe_items,1,"native preflight")
assert r[0]["x"].shape==r[0]["y"].shape
assert torch.isfinite(r[0]["y"]).all()
print("Native attention target capture from student prefix PASS")
del r


In [ ]:
#@title 6. Whole-model PROBE fidelity + interleaved CUDA speed, without AUDIT leakage
@torch.inference_mode()
def get_reference(items,limit):
    output=[]
    for start in range(0,min(limit,len(items)),BATCH):
        selected=items[start:min(start+BATCH,limit)]
        b=direct_batch(selected)
        with torch.autocast("cuda",dtype=amp_dtype):
            scores,_=teacher(**b)
        probs=torch.softmax(scores.float().masked_fill(
            ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()
        for j,item in enumerate(selected):
            output.append(dict(probs=probs[j],teacher=int(probs[j].argmax()),
                               gold=int(item["gold_index"]),
                               case=item["case_id"],qid=item["qid"]))
    return output

@torch.inference_mode()
def evaluate(plan,items,reference):
    n=len(reference)
    agree=correct=tcorrect=0;js=0.
    with install_plan(plan):
        for start in range(0,n,BATCH):
            selected=items[start:min(n,start+BATCH)]
            b=direct_batch(selected)
            with torch.autocast("cuda",dtype=amp_dtype):
                score,_=teacher(**b)
            p=torch.softmax(score.float().masked_fill(
                ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()
            for j,item in enumerate(selected):
                t=reference[start+j]["probs"];s=p[j]
                assert s.shape==t.shape
                m=(s+t)/2
                js+=float(.5*((s*(s.clamp_min(1e-8).log()-
                                    m.clamp_min(1e-8).log())).sum()+
                               (t*(t.clamp_min(1e-8).log()-
                                    m.clamp_min(1e-8).log())).sum()))
                guess=int(s.argmax())
                agree+=int(guess==reference[start+j]["teacher"])
                correct+=int(guess==item["gold_index"])
                tcorrect+=int(reference[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,
                gold_accuracy=correct/n,teacher_gold_accuracy=tcorrect/n,
                gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(before,after,repeats=BENCH_REPEATS):
    comparisons=[]
    for b in BENCH_BATCHES:
        def once(plan):
            with install_plan(plan),torch.autocast("cuda",dtype=amp_dtype):
                a=torch.cuda.Event(enable_timing=True)
                z=torch.cuda.Event(enable_timing=True)
                a.record();teacher(**b);z.record();z.synchronize()
                return float(a.elapsed_time(z))
        for _ in range(2):once(before);once(after)
        for step in range(repeats):
            readings={"a":[],"b":[]}
            order=("a","b","b","a") if step%2==0 else ("b","a","a","b")
            for label in order:
                readings[label].append(once(before if label=="a" else after))
            ma=float(np.mean(readings["a"]));mb=float(np.mean(readings["b"]))
            comparisons.append(dict(
                batch=int(b["input_ids"].shape[0]),
                tokens=int(b["input_ids"].shape[1]),
                before_ms=ma,after_ms=mb,speedup=ma/max(mb,1e-8)))
    return dict(median_speedup=float(np.median(
        [c["speedup"] for c in comparisons])),
        before_ms=float(np.median([c["before_ms"] for c in comparisons])),
        after_ms=float(np.median([c["after_ms"] for c in comparisons])),
        comparisons=comparisons)

PROBE_REF=get_reference(probe_items,PROBE_DECISIONS)
native_probe=evaluate({},probe_items,PROBE_REF)
assert native_probe["teacher_agreement"]==1 and native_probe["mean_js"]<1e-7
print("Original probe",native_probe)
print("Benchmark shapes",[(b["input_ids"].shape[0],b["input_ids"].shape[1])
                           for b in BENCH_BATCHES])


In [ ]:
#@title 7. Resume-safe experiment state and per-layer FP32 CeNN trainer
from datetime import datetime,timezone
PROGRESS_PATH=OUT/"v42_progress.json"
ADAPTER_DIR=OUT/"adapters"
ADAPTER_DIR.mkdir(parents=True,exist_ok=True)
signature=dict(
    source=SOURCE_MODEL,seed=SEED,
    train_cases=TRAIN_CASES,dev_cases=DEV_CASES,
    rank=RANK,steps_full=STEPS_FULL,
    steps_sliding=STEPS_SLIDING,selection_mode=SELECTION_MODE,
    learning_rate=LEARNING_RATE,
    probe_cases=[(x["case_id"],x["qid"]) for x in probe_items[:PROBE_DECISIONS]],
    experiment_version="v42-a1"
)
fingerprint=hashlib.sha256(
    json.dumps(signature,sort_keys=True).encode()).hexdigest()

if RESUME and PROGRESS_PATH.exists():
    progress=json.loads(PROGRESS_PATH.read_text())
    assert progress["fingerprint"]==fingerprint,(
        "Stored progress comes from a DIFFERENT run/configuration. "
        "Change OUT or settings to start a new experiment.")
    print("Restoring completed stages:",progress["completed"])
else:
    progress=dict(fingerprint=fingerprint,completed=[],accepted=[],
                  records=[],phase_a=None,finished=False)

def save_progress():
    tmp=PROGRESS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(progress,indent=2,default=str),
                   encoding="utf-8")
    tmp.replace(PROGRESS_PATH)

def restore_adapter(i):
    saved=ADAPTER_DIR/f"attention_{i:02d}.pt"
    payload=torch.load(saved,map_location="cpu",weights_only=True)
    assert payload["index"]==i and payload["fingerprint"]==fingerprint
    model=new_adapter(i)
    model.load_state_dict(payload["weights"],strict=True)
    model.eval().requires_grad_(False)
    return model

PLAN={}
for i in progress["accepted"]:
    PLAN[int(i)]=restore_adapter(int(i))
assert progress["accepted"]==[i for i in ORDER if i in PLAN]
save_progress()
print("Restored accepted layers:",list(PLAN))
print("State file:",PROGRESS_PATH,
      "(enable SAVE_TO_GOOGLE_DRIVE for persistence across runtime resets)")

def fit_single_layer(i,prefix):
    steps=STEPS_FULL if i in FULL else STEPS_SLIDING
    train_cache=capture_layer(
        i,prefix,train_items,LOCAL_TRAIN_BATCHES,"TRAIN")
    valid_cache=capture_layer(
        i,prefix,probe_items,LOCAL_PROBE_BATCHES,"PROBE")
    model=new_adapter(i)
    opt=torch.optim.AdamW(model.parameters(),
                          lr=LEARNING_RATE,weight_decay=1e-4)
    best=None;best_score=float("inf")
    best_weights=adapter_state(model)
    record=[]
    safe_weights=adapter_state(model)
    updates=attempts=recovery=0
    while updates<steps:
        attempts+=1
        if attempts>steps+40:
            print("Stopping after repeated numerical retries at layer",i)
            break
        source=to_cuda(train_cache[(attempts-1)%len(train_cache)])
        model.train();opt.zero_grad(set_to_none=True)
        loss,nmse,cos=mse_and_cosine(model,source)
        reason=None
        if not bool(torch.isfinite(loss.detach())):
            reason="nonfinite_forward"
        else:
            loss.backward()
            if any(p.grad is not None and
                   not bool(torch.isfinite(p.grad).all())
                   for p in model.parameters()):
                reason="nonfinite_grad"
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(),.5)
                opt.step()
                if any(not bool(torch.isfinite(p).all())
                       for p in model.parameters()):
                    reason="nonfinite_parameters"
        if reason:
            model.load_state_dict(safe_weights,strict=True)
            opt.state.clear()
            for group in opt.param_groups:group["lr"]*=.5
            recovery+=1
            print(" L",i,"recover from",reason,"LR",opt.param_groups[0]["lr"])
            if recovery>=8:break
            continue
        updates+=1
        if updates%10==0:safe_weights=adapter_state(model)
        if updates==1 or updates%LOG_EVERY==0 or updates==steps:
            quality=local_score(model,valid_cache)
            score=quality["nmse"]+.15*(1-quality["cosine"])
            record.append(dict(step=updates,quality=quality,
                               train_loss=float(loss.detach())))
            if score<best_score:
                best_score=score;best=quality
                best_weights=adapter_state(model)
            print(f" L{i:02d} {updates}/{steps} "
                  f"cos={quality['cosine']:.4f} "
                  f"NMSE={quality['nmse']:.4f} "
                  f"best={best['cosine']:.4f}")
    model.load_state_dict(best_weights,strict=True)
    model.eval().requires_grad_(False)
    assert all(bool(torch.isfinite(p).all()) for p in model.parameters())
    payload=dict(index=i,fingerprint=fingerprint,
                 phase="FULL" if i in FULL else "SLIDING",
                 steps=updates,recoveries=recovery,
                 best_local=best,training_history=record,
                 weights=adapter_state(model))
    torch.save(payload,ADAPTER_DIR/f"attention_{i:02d}.pt")
    del opt,best_weights,train_cache,valid_cache
    torch.cuda.empty_cache()
    return model,payload


In [ ]:
#@title 8. PHASE A: progressively train and replace ALL TEN FULL attention modules FIRST
def make_decision(i,pre,after,paired):
    gates=dict(
        teacher_agreement=after["teacher_agreement"]>=MIN_AGREEMENT,
        gold_loss=after["gold_drop"]<=MAX_GOLD_LOSS_PP/100,
        no_step_accuracy_regression=after["gold_accuracy"]>=pre["gold_accuracy"]-.01,
        speed=(paired["median_speedup"]>1.0),
    )
    eligible=(gates["teacher_agreement"] and gates["gold_loss"] and
              gates["no_step_accuracy_regression"])
    if REQUIRE_SPEEDUP_IN_GATED_MODE:eligible=eligible and gates["speed"]
    # The user-requested full 28-module CeNN experiment is explicitly
    # allowed in research_all, while exposing every failed safety gate.
    accepted=(SELECTION_MODE=="research_all" or eligible)
    return accepted,gates

def run_stage(i,phase):
    global PLAN
    assert i in ORDER
    assert phase==("FULL" if i in FULL else "SLIDING")
    if i in progress["completed"]:
        print("SKIP completed",phase,"layer",i,
              "accepted" if i in PLAN else "native retained")
        return
    before=evaluate(PLAN,probe_items,PROBE_REF)
    print(f"\n== {phase} L{i:02d}: training native-operator CeNN targets "
          f"on current accepted prefix (accepted={len(PLAN)}) ==")
    adapter,training=fit_single_layer(i,PLAN)
    proposed={**PLAN,i:adapter}
    after=evaluate(proposed,probe_items,PROBE_REF)
    pair=benchmark_paired(PLAN,proposed)
    accepted,gates=make_decision(i,before,after,pair)
    print(" Layer",i,"probe agreement",round(after["teacher_agreement"],4),
          "gold drop",round(after["gold_drop"],4),
          "speed",round(pair["median_speedup"],4),"x",
          "accepted",accepted,"gates",gates)
    if accepted:
        PLAN[i]=adapter
    progress["records"].append(dict(
        index=i,phase=phase,accepted=accepted,
        selection_mode=SELECTION_MODE,
        quality_gates=gates,
        before_probe=before,after_probe=after,
        paired_cuda=pair,local=training["best_local"],
        train_steps=training["steps"],
        recovery_events=training["recoveries"],
    ))
    progress["completed"].append(i)
    progress["accepted"]=[j for j in ORDER if j in PLAN]
    save_progress()
    if not accepted:del adapter
    torch.cuda.empty_cache()

# STRICT ORDER. No sliding CeNN is trained before this cell finishes.
for layer_id in FULL:
    run_stage(layer_id,"FULL")
assert all(i in progress["completed"] for i in FULL)
assert not any(i in progress["completed"] for i in SLIDING) or (
    progress["phase_a"] is not None
),"Completed sliding work requires a saved full-attention phase summary."

if progress.get("phase_a") is None:
    full_probe=evaluate(PLAN,probe_items,PROBE_REF)
    full_cuda=benchmark_paired({},PLAN,repeats=max(4,BENCH_REPEATS))
    progress["phase_a"]=dict(
        replaced=[i for i in FULL if i in PLAN],
        probe=full_probe,paired_cuda_vs_native=full_cuda,
        timestamp=datetime.now(timezone.utc).isoformat())
    save_progress()
print("\nPHASE A COMPLETE. Attempted 10 FULL layers, accepted",
      len(progress["phase_a"]["replaced"]))
print("FULL-ONLY PROBE results:",progress["phase_a"]["probe"])
print("FULL-ONLY speedup:",progress["phase_a"]["paired_cuda_vs_native"]["median_speedup"])
print("Now proceed to PHASE B, sliding layers.")


In [ ]:
#@title 9. PHASE B: progressively train and replace all 18 SLIDING attention modules
assert progress.get("phase_a") is not None,(
    "Run the entire FULL-attention phase (cell 8) first.")
assert all(i in progress["completed"] for i in FULL)

for layer_id in SLIDING:
    run_stage(layer_id,"SLIDING")
assert len(progress["completed"])==28
assert progress["completed"]==list(ORDER),(
    "Completed layer ordering differs from required full-then-sliding order.",
    progress["completed"])
final_probe=evaluate(PLAN,probe_items,PROBE_REF)
progress["final_probe"]=final_probe
progress["final_plan_indices"]=[i for i in ORDER if i in PLAN]
save_progress()
print("\nPHASE B COMPLETE. All 28 attention modules individually trained/attempted.")
print("Accepted FULL",len([i for i in FULL if i in PLAN]),
      "SLIDING",len([i for i in SLIDING if i in PLAN]))
print("End of training PROBE:",final_probe)
print("ATTENTION modules modified:",
      [i for i in ORDER if i in PLAN])
print("Native residuals and original FFNs retained for ALL 28 layers.")


In [ ]:
#@title 10. Implementation
# Additional code follows in the next commit.


In [ ]:
#@title 11. Implementation
# Additional code follows in the next commit.
